# Section 5 Implementing a ResNet-34 CNN Using Keras
##### Residual Unit

In [ ]:
from typing import Sequence, Union

import tensorflow as tf

class ResidualUnit(tf.keras.layers.Layer):
    def __init__(self, filters: int, kernel_size: Union[Sequence[int], int] = 3, stride: int = 1, activation: Union[tf.keras.layers.Activation, str] = "relu", **kwargs) -> None:
        super().__init__(**kwargs)
        self.main_block = [
            tf.keras.layers.Conv2D(filters, kernel_size=kernel_size, strides=stride, padding="same"),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Conv2D(filters, kernel_size=kernel_size, strides=1, padding="same"),
            tf.keras.layers.BatchNormalization(),
        ]

        self.res_block = [] if stride == 1 else \
            [
                tf.keras.layers.Conv2D(filters, kernel_size=(1, 1), strides=stride, padding="same"),
                tf.keras.layers.BatchNormalization()
            ]

        self.activation = tf.keras.activations.get(activation)

    def call(self, inputs: Sequence[Sequence[Sequence[float]]], *args, **kwargs) -> tf.Tensor:
        z_main = inputs
        for layer in self.main_block:
            z_main = layer(z_main)

        z_res = inputs
        for layer in self.res_block:
            z_res = layer(z_res)

        return self.activation(z_main + z_res)

2022-11-22 07:32:30.565967: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


##### ResNet-34

In [3]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(64, 7, 2, padding="same", activation="relu", use_bias=False, input_shape=(224, 224, 3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPool2D(3, 2, padding="same"),
    *[ResidualUnit(filters=64, kernel_size=3, stride=1, activation="relu") for _ in range(3)],
    ResidualUnit(filters=128, kernel_size=3, stride=2, activation="relu"),
    *[ResidualUnit(filters=128, kernel_size=3, stride=1, activation="relu") for _ in range(3)],
    ResidualUnit(filters=256, kernel_size=3, stride=2, activation="relu"),
    *[ResidualUnit(filters=256, kernel_size=3, stride=1, activation="relu") for _ in range(5)],
    ResidualUnit(filters=512, kernel_size=3, stride=2, activation="relu"),
    *[ResidualUnit(filters=512, kernel_size=3, stride=1, activation="relu") for _ in range(2)],
    tf.keras.layers.GlobalAvgPool2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1000, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics="accuracy")
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_36 (Conv2D)          (None, 112, 112, 64)      9408      
                                                                 
 batch_normalization_36 (Bat  (None, 112, 112, 64)     256       
 chNormalization)                                                
                                                                 
 activation (Activation)     (None, 112, 112, 64)      0         
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 56, 56, 64)       0         
 2D)                                                             
                                                                 
 residual_unit_16 (ResidualU  (None, 56, 56, 64)       74368     
 nit)                                                            
                                                      